In [2]:
import json
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import time
import random
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast

In [ ]:
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 64
EPOCHS      = 60
NUM_CLASSES = 300
JSON_FILE   = ""
SAVE_ROOT   = ""
CHECKPOINT  = ""
os.makedirs(os.path.dirname(CHECKPOINT), exist_ok=True)

%load_ext autoreload 
%autoreload 2

In [4]:
from utils.lstm import FeatureLSTM, FeatureDataset

In [5]:
def run_epoch(model, loader, optimizer, criterion, scaler,
              device, epoch, total_epochs, train=True):
    model.train() if train else model.eval()

    total_loss, correct, total = 0.0, 0, 0
    total_time, latencies      = 0.0, []

    tag  = "Train" if train else "Val"
    pbar = tqdm(loader, desc=f"{tag} [{epoch+1}/{total_epochs}]", leave=False)

    for feat, labels in pbar:
        feat, labels = feat.to(device), labels.to(device)
        t0 = time.perf_counter()

        if train:
            optimizer.zero_grad()
            with autocast('cuda'):
                out  = model(feat)
                loss = criterion(out, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            with torch.no_grad():
                with autocast('cuda'):
                    out  = model(feat)
                    loss = criterion(out, labels)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            t1          = time.perf_counter()
            batch_time  = t1 - t0
            total_time += batch_time
            latencies.append(batch_time / feat.size(0) * 1000)

        total_loss += loss.item()
        _, pred = out.max(1)
        total   += labels.size(0)
        correct += pred.eq(labels).sum().item()
        pbar.set_postfix(loss=f"{loss.item():.3f}",
                         acc=f"{100.*correct/total:.1f}%")

    avg_loss = total_loss / len(loader)
    acc      = 100. * correct / total
    avg_lat  = float(np.mean(latencies)) if latencies else 0.0
    fps      = total / total_time        if total_time > 0 else 0.0

    if not train:
        print(f"Val Acc: {acc:.2f} | Latency: {avg_lat:.2f} ms/video | FPS: {fps:.1f}")

    return avg_loss, acc, avg_lat, fps

In [6]:

if __name__ == "__main__":
    model     = FeatureLSTM(input_size=768, hidden=256, num_layers=1,
                            num_classes=NUM_CLASSES).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    train_set = FeatureDataset(JSON_FILE, SAVE_ROOT, split='train')
    val_set   = FeatureDataset(JSON_FILE, SAVE_ROOT, split='val',
                               label_map=train_set.action_to_idx)
    test_set  = FeatureDataset(JSON_FILE, SAVE_ROOT, split='test',
                               label_map=train_set.action_to_idx)

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    print(f"Device: {DEVICE}")

    model     = FeatureLSTM(input_size=768, hidden=256, num_layers=1,
                            num_classes=NUM_CLASSES).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler    = GradScaler('cuda')

    best_val_acc = 0.0

    for epoch in range(EPOCHS):
        t0 = time.time()
        train_loss, train_acc, _, _ = run_epoch(
            model, train_loader, optimizer, criterion, scaler,
            DEVICE, epoch, EPOCHS, train=True)
        val_loss, val_acc, val_lat, val_fps = run_epoch(
            model, val_loader, optimizer, criterion, scaler,
            DEVICE, epoch, EPOCHS, train=False)
        scheduler.step()

        print(f"Epoch [{epoch+1:02d}/{EPOCHS}] "
              f"Train Loss {train_loss:.4f} Acc {train_acc:.2f}% | "
              f"Val Loss {val_loss:.4f} Acc {val_acc:.2f}% | "
              f"{time.time()-t0:.1f}s")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'val_acc': val_acc,
                'label_map': train_set.action_to_idx,
            }, CHECKPOINT)
            print(f"✅ Best model saved (val acc {val_acc:.2f}%)")

    print("\n" + "="*50)
    ckpt = torch.load(CHECKPOINT)
    model.load_state_dict(ckpt['model_state_dict'])
    _, test_acc, test_lat, test_fps = run_epoch(
        model, test_loader, None, criterion, scaler,
        DEVICE, 0, 1, train=False)
    print(f"Final Test Acc: {test_acc:.2f}% | "
          f"Latency: {test_lat:.2f} ms/video | FPS: {test_fps:.1f}")

[TRAIN] 0 videos | 300 classes
[VAL] 0 videos | 300 classes
[TEST] 0 videos | 300 classes


ValueError: num_samples should be a positive integer value, but got num_samples=0